**Name:** Siddhi Shashikant Gandhi

**Roll No.** 33

**Batch** T2

In [1]:
!pip install pandas numpy scikit-learn tensorflow

**Import Libraries**

In [2]:
import pandas as pd                      # For data handling
import numpy as np                       # For numerical operations

from sklearn.model_selection import train_test_split   # Split dataset
from sklearn.preprocessing import LabelEncoder         # Convert labels to numbers

from tensorflow.keras.preprocessing.text import Tokenizer   # Convert text → numbers
from tensorflow.keras.preprocessing.sequence import pad_sequences  # Make equal length

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, GRU, Dense

**Load Dataset**

In [3]:
# Load SMS spam dataset from online source
url = 'https://raw.githubusercontent.com/justmarkham/pycon-2016-tutorial/master/data/sms.tsv'

# Read dataset (tab-separated file)
data = pd.read_csv(url, sep='\t', header=None, names=['label', 'message'])

# Display first 5 rows
data.head()

,label,message
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


In [4]:
data.shape

(5572, 2)

In [5]:
data.sample(10)

,label,message
2825,ham,No need to buy lunch for me.. I eat maggi mee..
294,ham,Are you this much buzy
2988,ham,"I'm there and I can see you, but you can't see..."
3125,ham,My uncles in Atlanta. Wish you guys a great se...
547,ham,I know but you need to get hotel now. I just g...
5184,ham,I'm in town now so i'll jus take mrt down later.
5484,ham,", , and picking them up from various points ..."
3796,ham,Also remember the beads don't come off. Ever.
2101,ham,Oh Howda gud gud.. Mathe en samachara chikku:-)
5359,ham,This is ur face test ( 1 2 3 4 5 6 7 8 9 &lt;...


 **Encode Labels**

In [6]:
# Convert text labels into numeric format
le = LabelEncoder()

# spam → 1, ham → 0
data['label'] = le.fit_transform(data['label'])

 **Tokenization**

In [7]:
data.head()

,label,message
0,0,"Go until jurong point, crazy.. Available only ..."
1,0,Ok lar... Joking wif u oni...
2,1,Free entry in 2 a wkly comp to win FA Cup fina...
3,0,U dun say so early hor... U c already then say...
4,0,"Nah I don't think he goes to usf, he lives aro..."


In [8]:
# Create tokenizer object
tokenizer = Tokenizer()

# Learn vocabulary from text
tokenizer.fit_on_texts(data['message'])

# Convert sentences into sequences of numbers
sequences = tokenizer.texts_to_sequences(data['message'])

**How a Message Converts to Sequence?**

In [9]:
# Select one sample message
sample_text = data['message'][0]

print("Original Message:\n", sample_text)

# Convert to sequence
sample_sequence = tokenizer.texts_to_sequences([sample_text])

print("\nConverted Sequence:\n", sample_sequence)

# Show word index mapping
print("\nWord Index Mapping (first 10 words):")
for word, index in list(tokenizer.word_index.items())[:10]:
    print(f"{word} → {index}")

Original Message:
 Go until jurong point, crazy.. Available only in bugis n great world la e buffet... Cine there got amore wat...

Converted Sequence:
 [[49, 471, 4436, 842, 755, 658, 64, 8, 1327, 88, 123, 351, 1328, 148, 2997, 1329, 67, 58, 4437, 144]]

Word Index Mapping (first 10 words):
i → 1
to → 2
you → 3
a → 4
the → 5
u → 6
and → 7
in → 8
is → 9
me → 10


**Padding**

In [10]:
# Define maximum length of sequence
max_len = 100

# Pad sequences so all have same length
X = pad_sequences(sequences, maxlen=max_len)

# Target variable
y = data['label']

**Sample After Padding**

In [11]:
X = pad_sequences(sequences, maxlen=max_len)
# Take the same sample text
sample_text = data['message'][0]

print("Original Message:\n", sample_text)

# Convert to sequence
sample_sequence = tokenizer.texts_to_sequences([sample_text])
print("\nSequence Before Padding:\n", sample_sequence)

# Apply padding
sample_padded = pad_sequences(sample_sequence, maxlen=max_len)
print("\nSequence After Padding:\n", sample_padded)

# Show length comparison
print("\nLength Before Padding:", len(sample_sequence[0]))
print("Length After Padding:", len(sample_padded[0]))

Original Message:
 Go until jurong point, crazy.. Available only in bugis n great world la e buffet... Cine there got amore wat...

Sequence Before Padding:
 [[49, 471, 4436, 842, 755, 658, 64, 8, 1327, 88, 123, 351, 1328, 148, 2997, 1329, 67, 58, 4437, 144]]

Sequence After Padding:
 [[   0    0    0    0    0    0    0    0    0    0    0    0    0    0
     0    0    0    0    0    0    0    0    0    0    0    0    0    0
     0    0    0    0    0    0    0    0    0    0    0    0    0    0
     0    0    0    0    0    0    0    0    0    0    0    0    0    0
     0    0    0    0    0    0    0    0    0    0    0    0    0    0
     0    0    0    0    0    0    0    0    0    0   49  471 4436  842
   755  658   64    8 1327   88  123  351 1328  148 2997 1329   67   58
  4437  144]]

Length Before Padding: 20
Length After Padding: 100


**Train-Test Split**

In [12]:
# Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

**Build GRU Model**

In [13]:
# Vocabulary size
vocab_size = len(tokenizer.word_index) + 1

# Build model
model = Sequential([

    # Convert word indices → dense vectors
    Embedding(input_dim=vocab_size, output_dim=64, input_length=max_len),

    # GRU layer (captures sequence information)
    GRU(64),

    # Output layer (binary classification)
    Dense(1, activation='sigmoid')
])

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


 **Compile Model**

In [14]:
# Configure model training
model.compile(
    loss='binary_crossentropy',   # For binary classification
    optimizer='adam',             # Optimization algorithm
    metrics=['accuracy']          # Evaluation metric
)

**Model Summary**

In [15]:
# Display model architecture
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru (GRU)                       │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

**Train Model**

In [16]:
model.fit(X_train, y_train, epochs=5, batch_size=32, validation_data=(X_test, y_test))

Epoch 1/5
140/140 ━━━━━━━━━━━━━━━━━━━━ 13s 72ms/step - accuracy: 0.9311 - loss: 0.1934 - val_accuracy: 0.9785 - val_loss: 0.0758
Epoch 2/5
140/140 ━━━━━━━━━━━━━━━━━━━━ 9s 64ms/step - accuracy: 0.9928 - loss: 0.0243 - val_accuracy: 0.9830 - val_loss: 0.0624
Epoch 3/5
140/140 ━━━━━━━━━━━━━━━━━━━━ 10s 74ms/step - accuracy: 0.9984 - loss: 0.0104 - val_accuracy: 0.9839 - val_loss: 0.0671
Epoch 4/5
140/140 ━━━━━━━━━━━━━━━━━━━━ 10s 73ms/step - accuracy: 0.9993 - loss: 0.0031 - val_accuracy: 0.9848 - val_loss: 0.0931
Epoch 5/5
140/140 ━━━━━━━━━━━━━━━━━━━━ 10s 72ms/step - accuracy: 0.9998 - loss: 9.3856e-04 - val_accuracy: 0.9839 - val_loss: 0.0835


**Evaluate Model**

In [17]:
# Evaluate on test data
loss, acc = model.evaluate(X_test, y_test)
print('Accuracy:', acc)

35/35 ━━━━━━━━━━━━━━━━━━━━ 1s 21ms/step - accuracy: 0.9839 - loss: 0.0835
Accuracy: 0.9838564991950989


**Save the Trained Model**

In [18]:
# Save the trained model
model.save("GRU_Spam_Detector.h5")

print("Model saved successfully!")

Model saved successfully!


In [19]:
#to avoids legacy( .h5) issues, new .keras format is recommended
model.save("gru_model.keras")

**Save Tokenizer**

In [20]:
import pickle

# Save tokenizer
with open("GRU_Tokenizer.pkl", "wb") as f:
    pickle.dump(tokenizer, f)

print(" Tokenizer saved successfully!")

 Tokenizer saved successfully!


**Test with Custom Message (User Input)**

In [21]:
# Function to predict whether a message is Spam or Ham
def predict_message(text):

    # Convert text → sequence using trained tokenizer
    seq = tokenizer.texts_to_sequences([text])

    # Apply padding (same length as training)
    padded = pad_sequences(seq, maxlen=max_len)

    # Predict using trained model
    prediction = model.predict(padded)[0][0]

    # Print result
    print("\nMessage:", text)
    print("Spam Probability:", prediction)

    # Classification based on threshold
    if prediction > 0.5:
        print("Result:  SPAM")
    else:
        print("Result:  HAM (Not Spam)")

In [22]:
# Take input from user
user_input = input("Enter a message to test: ")

# Predict
predict_message(user_input)

Enter a message to test: hii
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 221ms/step

Message: hii
Spam Probability: 0.0024787998
Result:  HAM (Not Spam)


In [23]:
test_messages = [
    "Free entry in a contest",
    "Call me when you reach home",
    "You have won a prize!!!"
]

for msg in test_messages:
    predict_message(msg)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step

Message: Free entry in a contest
Spam Probability: 0.04181931
Result:  HAM (Not Spam)
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step

Message: Call me when you reach home
Spam Probability: 7.4587304e-05
Result:  HAM (Not Spam)
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 42ms/step

Message: You have won a prize!!!
Spam Probability: 0.99618155
Result:  SPAM


**app.py**

In [ ]:
from flask import Flask, render_template, request
import numpy as np
import pickle
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing.sequence import pad_sequences

# Initialize app
app = Flask(__name__)

# Load model
model = load_model("model.h5", compile=False)

# Load tokenizer
with open("tokenizer.pkl", "rb") as f:
    tokenizer = pickle.load(f)

# IMPORTANT: same as training
MAX_LEN = 100

@app.route("/")
def home():
    return render_template("index.html")

@app.route("/predict", methods=["POST"])
def predict():
    text = request.form["text"]

    # Convert text → sequence
    sequence = tokenizer.texts_to_sequences([text])

    # Padding
    padded = pad_sequences(sequence, maxlen=MAX_LEN)

    # Prediction
    prediction = model.predict(padded)[0][0]

    # Result
    if prediction > 0.5:
        result = "Positive 😊"
    else:
        result = "Negative 😠"

    return render_template("index.html", prediction=result, text=text)

if __name__ == "__main__":
    app.run(debug=True)

**index.html**

In [ ]:
<!DOCTYPE html>
<html>
<head>
    <title>Sentiment Analysis GRU</title>
</head>
<body style="text-align:center; font-family:Arial;">

    <h1>GRU Sentiment Analysis</h1>

    <form action="/predict" method="post">
        <textarea name="text" rows="5" cols="40" placeholder="Enter text"></textarea><br><br>
        <button type="submit">Predict</button>
    </form>

    {% if prediction %}
        <h2>Result: {{ prediction }}</h2>
        <p>{{ text }}</p>
    {% endif %}

</body>
</html>